# Probe System With Custom Input Sequence

Use this notebook to run the inverted pendulum with a user-defined force sequence.

Workflow:
1. Configure simulation and initial state.
2. Define your custom input sequence `u_custom` over `t_custom`.
3. Run the simulation.
4. Inspect outputs and visualize states/input + animation.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

sys.path.append(str(Path("../src").resolve()))

from config import PendulumParams, SimulationSettings
from initial_state import derive_initial_state_from_impulse_crossing
from inputs import PiecewiseLinearInput
from simulation import run_simulation
from visualization import plot_states_and_input, animate_cart_pendulum

In [ ]:
# Simulation configuration
params = PendulumParams(M=1.0, m=0.2, l=0.5, g=9.81)
sim_settings = SimulationSettings(t_end=10.0, num_samples=1000)

# Initial-state selection
use_derived_initial_state = True
theta_target_deg = 10.0
impulse_force = 0.01

if use_derived_initial_state:
    init = derive_initial_state_from_impulse_crossing(
        theta_target_deg=theta_target_deg,
        params=params,
        settings=sim_settings,
        impulse_force=impulse_force,
    )
    if not init.success:
        raise RuntimeError(f"Failed to derive initial state: {init.diagnostic}")
    y0 = init.initial_state
else:
    # Manual state: [x, x_dot, theta(rad), theta_dot(rad/s)]
    y0 = np.array([0.0, 0.0, np.deg2rad(10.0), 0.0], dtype=float)

print("Initial state y0:", y0)

In [ ]:
# Define custom input sequence u_custom over t_custom.
# Replace this with any sequence you want to probe.

t_custom = np.linspace(0.0, sim_settings.t_end, sim_settings.num_samples)

# Example profile: piecewise-interpolated force sequence
time_knots = np.array([0.0, 1.0, 2.5, 4.0, 6.0, 8.0, 10.0], dtype=float)
force_knots = np.array([0.0, 0.2, -0.1, 0.3, -0.25, 0.1, 0.0], dtype=float)
u_custom = np.interp(t_custom, time_knots, force_knots)

# Validation
if u_custom.shape[0] != t_custom.shape[0]:
    raise ValueError("u_custom must have the same length as t_custom")
if not np.all(np.isfinite(u_custom)):
    raise ValueError("u_custom contains non-finite values")

print("Custom input length:", u_custom.shape[0])
print("Input min/max:", float(np.min(u_custom)), float(np.max(u_custom)))